# 3. Özellik Mühendisliği

Bu notebook'ta meteorolojik verilerden özellikler çıkaracak ve modele hazır veri seti oluşturacağız.

**İçerik:**
- Meteorolojik özelliklerin eklenmesi
- Lag (gecikme) özellikleri
- Rolling window istatistikleri
- Türetilmiş özellikler

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
print('Kütüphaneler yüklendi!')

## 3.1 Verilerin Yüklenmesi

In [ ]:
DATA_PATH = '../data/'

ana_df = pd.read_pickle(DATA_PATH + 'ana_veri_seti.pkl')
istasyonlar = pd.read_pickle(DATA_PATH + 'istasyonlar.pkl')

print(f"Ana veri seti: {ana_df.shape}")
print(f"\nSütunlar: {ana_df.columns.tolist()}")

## 3.2 İstasyon Bilgilerinin Eklenmesi

In [ ]:
# İstasyon bilgilerini ekle
ana_df = ana_df.merge(
    istasyonlar[['istasyon_no', 'enlem', 'boylam', 'rakim']],
    on='istasyon_no',
    how='left'
)

print("İstasyon bilgileri eklendi!")
display(ana_df.head())

## 3.3 Lag (Gecikme) Özellikleri

In [ ]:
def ekle_lag_ozellikleri(df, sutun, lag_gunleri=[1, 2, 3, 7]):
    """Belirtilen sütun için lag özellikleri ekler."""
    for lag in lag_gunleri:
        df[f'{sutun}_lag{lag}'] = df.groupby('istasyon_no')[sutun].shift(lag)
    return df

# Yıldırım sayısı için lag özellikleri
ana_df = ekle_lag_ozellikleri(ana_df, 'yildirim_sayisi', [1, 2, 3, 7])
ana_df = ekle_lag_ozellikleri(ana_df, 'yildirim_var', [1, 2, 3, 7])

print("Lag özellikleri eklendi!")

## 3.4 Rolling Window İstatistikleri

In [ ]:
def ekle_rolling_ozellikler(df, sutun, pencereler=[3, 7, 14]):
    """Rolling window istatistikleri ekler."""
    for pencere in pencereler:
        # Ortalama
        df[f'{sutun}_rolling_mean_{pencere}'] = df.groupby('istasyon_no')[sutun].transform(
            lambda x: x.rolling(window=pencere, min_periods=1).mean().shift(1)
        )
        # Toplam
        df[f'{sutun}_rolling_sum_{pencere}'] = df.groupby('istasyon_no')[sutun].transform(
            lambda x: x.rolling(window=pencere, min_periods=1).sum().shift(1)
        )
    return df

# Yıldırım için rolling özellikler
ana_df = ekle_rolling_ozellikler(ana_df, 'yildirim_sayisi', [3, 7, 14])

print("Rolling özellikler eklendi!")

## 3.5 Döngüsel Özellikler

In [ ]:
# Ay ve gün için döngüsel (sinüs/kosinüs) kodlama
ana_df['ay_sin'] = np.sin(2 * np.pi * ana_df['ay'] / 12)
ana_df['ay_cos'] = np.cos(2 * np.pi * ana_df['ay'] / 12)

ana_df['gun_sin'] = np.sin(2 * np.pi * ana_df['yilin_gunu'] / 365)
ana_df['gun_cos'] = np.cos(2 * np.pi * ana_df['yilin_gunu'] / 365)

ana_df['hafta_sin'] = np.sin(2 * np.pi * ana_df['haftanin_gunu'] / 7)
ana_df['hafta_cos'] = np.cos(2 * np.pi * ana_df['haftanin_gunu'] / 7)

print("Döngüsel özellikler eklendi!")

## 3.6 Özellik Özeti

In [ ]:
print(f"Toplam özellik sayısı: {len(ana_df.columns)}")
print(f"\nTüm sütunlar:")
for i, col in enumerate(ana_df.columns, 1):
    print(f"  {i}. {col}")

In [ ]:
# Eksik değer kontrolü
eksik_degerler = ana_df.isnull().sum()
eksik_sutunlar = eksik_degerler[eksik_degerler > 0]

print("Eksik değer içeren sütunlar:")
for col, count in eksik_sutunlar.items():
    print(f"  {col}: {count} ({count/len(ana_df)*100:.2f}%)")

In [ ]:
# Eksik değerleri temizle (ilk günlerin lag değerleri için)
print(f"Temizlemeden önce: {len(ana_df)} satır")
ana_df_temiz = ana_df.dropna()
print(f"Temizlemeden sonra: {len(ana_df_temiz)} satır")
print(f"Silinen satır: {len(ana_df) - len(ana_df_temiz)}")

## 3.7 Korelasyon Analizi

In [ ]:
# Hedef değişkenle korelasyon
sayisal_sutunlar = ana_df_temiz.select_dtypes(include=[np.number]).columns.tolist()
if 'yildirim_var' in sayisal_sutunlar:
    korelasyonlar = ana_df_temiz[sayisal_sutunlar].corr()['yildirim_var'].sort_values(ascending=False)
    
    print("Hedef değişkenle en yüksek korelasyonlar:")
    print(korelasyonlar.head(15))
    print("\nEn düşük korelasyonlar:")
    print(korelasyonlar.tail(10))

In [ ]:
# Korelasyon görselleştirmesi
fig, ax = plt.subplots(figsize=(12, 8))

# En önemli özellikleri seç
onemli_ozellikler = ['yildirim_var', 'yildirim_var_lag1', 'yildirim_var_lag2', 
                     'yildirim_sayisi_rolling_sum_7', 'ay', 'mevsim', 'rakim']
mevcut_ozellikler = [f for f in onemli_ozellikler if f in ana_df_temiz.columns]

if len(mevcut_ozellikler) > 1:
    kor_matris = ana_df_temiz[mevcut_ozellikler].corr()
    sns.heatmap(kor_matris, annot=True, cmap='RdBu_r', center=0, ax=ax)
    ax.set_title('Özellik Korelasyon Matrisi')
    plt.tight_layout()
    plt.savefig('../tez_docs/figures/korelasyon_matrisi.png', dpi=150)
    plt.show()

## 3.8 Verilerin Kaydedilmesi

In [ ]:
# Model için hazır veri setini kaydet
ana_df_temiz.to_pickle(DATA_PATH + 'model_veri_seti.pkl')

print("Model için hazır veri seti kaydedildi!")
print(f"Boyut: {ana_df_temiz.shape}")

In [ ]:
# Özellik listesi
hedef = 'yildirim_var'
cikartilacaklar = ['tarih', 'yildirim_sayisi', 'yildirim_var', 'ort_akim', 'maks_akim', 'min_akim', 'ort_mesafe']
ozellikler = [col for col in ana_df_temiz.columns if col not in cikartilacaklar]

print(f"\nHedef değişken: {hedef}")
print(f"Özellik sayısı: {len(ozellikler)}")
print(f"\nÖzellikler:")
for i, f in enumerate(ozellikler, 1):
    print(f"  {i}. {f}")

# Özellik listesini kaydet
import json
with open(DATA_PATH + 'ozellik_listesi.json', 'w', encoding='utf-8') as f:
    json.dump({'hedef': hedef, 'ozellikler': ozellikler}, f, ensure_ascii=False, indent=2)

---
**Sonraki Adım:** `04_model_egitimi.ipynb` - Model Eğitimi